# Kronos — entrega.ipynb
## Validación y análisis del agente Connect-4

Este notebook genera todas las gráficas para la validación y optimización de Kronos.
Corre cada celda en orden. La primera celda de setup puede tomar algunos minutos.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

# Importar código base del torneo
sys.path.insert(0, os.path.abspath('../..'))
from connect4.connect_state import ConnectState

from policy import (
    Kronos, SharedQTable, train_kronos, run_self_play_episode,
    ROWS, COLS, get_free_cols, apply_move, check_winner, is_terminal,
    board_to_key, DEFAULT_SELF_PLAY_EPISODES
)

SEED = 42
np.random.seed(SEED)
random_rng = np.random.default_rng(SEED)
print('Setup OK')

---
## Utilidades compartidas

In [ ]:
def random_policy(board):
    free = get_free_cols(board)
    return int(np.random.choice(free))


def play_game(policy_red, policy_yellow, max_moves=42):
    """
    Juega una partida completa.
    policy_red / policy_yellow: callables (board) -> col
    Retorna: winner in {-1, 0, 1}
    """
    board = np.zeros((ROWS, COLS), dtype=int)
    player = -1
    for _ in range(max_moves):
        if is_terminal(board):
            break
        if player == -1:
            col = policy_red(board)
        else:
            col = policy_yellow(board)
        board = apply_move(board, col, player)
        player = -player
    return check_winner(board)


def evaluate(kronos_agent, opponent_fn, n_games=200, kronos_color=-1):
    """
    Evalúa Kronos contra un oponente.
    kronos_color: -1=Rojo (primer turno), 1=Amarillo (segundo turno)
    Retorna: (wins, draws, losses)
    """
    wins, draws, losses = 0, 0, 0
    kronos_fn = lambda b: kronos_agent.act(b)
    for _ in range(n_games):
        if kronos_color == -1:
            winner = play_game(kronos_fn, opponent_fn)
        else:
            winner = play_game(opponent_fn, kronos_fn)
        if winner == kronos_color:
            wins += 1
        elif winner == 0:
            draws += 1
        else:
            losses += 1
    return wins, draws, losses


print('Utilidades OK')

---
## 1. Efecto de los episodios de entrenamiento sobre el desempeño

Variable: número de episodios offline (recursos de entrenamiento)  
Oponente: jugador aleatorio  
Colores: rojo (primer turno) y amarillo (segundo turno)

In [ ]:
episode_counts = [200, 500, 1000, 2000, 4000, 6000]
N_EVAL = 200

results_vs_random = {'red': [], 'yellow': []}

for ep in episode_counts:
    print(f'Entrenando con {ep} episodios...')
    q = train_kronos(episodes=ep, save_path=None, verbose=False)
    agent = Kronos(episodes=ep)
    agent.q_table = q

    # Como rojo
    w, d, l = evaluate(agent, random_policy, n_games=N_EVAL, kronos_color=-1)
    results_vs_random['red'].append({'win': w/N_EVAL, 'draw': d/N_EVAL, 'loss': l/N_EVAL})

    # Como amarillo
    w, d, l = evaluate(agent, random_policy, n_games=N_EVAL, kronos_color=1)
    results_vs_random['yellow'].append({'win': w/N_EVAL, 'draw': d/N_EVAL, 'loss': l/N_EVAL})
    print(f'  Rojo: {w}/{N_EVAL} wins | Amarillo: {w}/{N_EVAL} wins')

# --- Gráfica ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fig.suptitle('Kronos vs. Jugador Aleatorio\nDesempeño en función de episodios de entrenamiento', fontsize=13)

for ax, color, label in zip(axes, ['red', 'yellow'], ['Rojo (primer turno)', 'Amarillo (segundo turno)']):
    wins  = [r['win']  for r in results_vs_random[color]]
    draws = [r['draw'] for r in results_vs_random[color]]
    losses= [r['loss'] for r in results_vs_random[color]]
    ax.stackplot(episode_counts, losses, draws, wins,
                 labels=['Derrota', 'Empate', 'Victoria'],
                 colors=['#e74c3c', '#f39c12', '#2ecc71'], alpha=0.85)
    ax.axhline(0.5, color='black', linestyle='--', linewidth=1, label='50% umbral mínimo')
    ax.set_title(label)
    ax.set_xlabel('Episodios de entrenamiento')
    ax.set_ylabel('Proporción de partidas')
    ax.legend(loc='lower right', fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('fig1_episodes_vs_random.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 1 guardada.')

---
## 2. Auto-desempeño: Kronos vs. Kronos

Se evalúa qué pasa cuando Kronos juega contra sí mismo.  
Según la teoría (slide 12): en un Alternating Markov Game de suma cero simétrico,  
el primer jugador tiene ventaja estructural.

In [ ]:
self_play_results = []

for ep in episode_counts:
    print(f'Entrenando {ep} episodios para auto-evaluación...')
    q = train_kronos(episodes=ep, save_path=None, verbose=False)

    agent1 = Kronos(); agent1.q_table = q
    agent2 = Kronos(); agent2.q_table = q  # misma política, instancias separadas

    wins1, draws, wins2 = 0, 0, 0
    for _ in range(N_EVAL):
        w = play_game(agent1.act, agent2.act)
        if w == -1: wins1 += 1
        elif w == 1: wins2 += 1
        else: draws += 1

    self_play_results.append({'wins1': wins1/N_EVAL, 'draws': draws/N_EVAL, 'wins2': wins2/N_EVAL})

# --- Gráfica ---
fig, ax = plt.subplots(figsize=(8, 5))
w1 = [r['wins1'] for r in self_play_results]
d  = [r['draws'] for r in self_play_results]
w2 = [r['wins2'] for r in self_play_results]

ax.stackplot(episode_counts, w2, d, w1,
             labels=['Gana Amarillo (2do)', 'Empate', 'Gana Rojo (1ro)'],
             colors=['#f1c40f', '#bdc3c7', '#e74c3c'], alpha=0.85)
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_title('Kronos vs. Kronos (auto-desempeño)\nen función de episodios de entrenamiento', fontsize=12)
ax.set_xlabel('Episodios de entrenamiento')
ax.set_ylabel('Proporción de partidas')
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig2_self_play.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 2 guardada.')

---
## 3. Efecto del Reward Shaping

Comparación entre Kronos con y sin reward shaping,  
evaluado contra el jugador aleatorio con 6000 episodios.

In [ ]:
shaping_weights = [0.0, 0.02, 0.05, 0.1, 0.2]
shaping_results = {'red': [], 'yellow': []}
EP = 3000

for sw in shaping_weights:
    print(f'shaping_weight={sw}...')
    q = train_kronos(episodes=EP, shaping_weight=sw, save_path=None, verbose=False)
    agent = Kronos(); agent.q_table = q

    for color_key, kronos_color in [('red', -1), ('yellow', 1)]:
        w, d, l = evaluate(agent, random_policy, n_games=N_EVAL, kronos_color=kronos_color)
        shaping_results[color_key].append(w / N_EVAL)

# --- Gráfica ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(shaping_weights, shaping_results['red'], 'o-', color='#e74c3c', label='Rojo (1ro)', linewidth=2)
ax.plot(shaping_weights, shaping_results['yellow'], 's--', color='#f39c12', label='Amarillo (2do)', linewidth=2)
ax.axhline(0.5, color='gray', linestyle=':', label='Umbral mínimo (50%)')
ax.set_title(f'Efecto del Reward Shaping sobre el desempeño\n({EP} episodios, vs. aleatorio)', fontsize=12)
ax.set_xlabel('Peso del reward shaping (λ)')
ax.set_ylabel('Tasa de victorias')
ax.legend()
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('fig3_shaping.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 3 guardada.')

---
## 4. Resumen: desempeño vs. recursos (gráfica tipo scatter)

Muestra múltiples variantes de Kronos en el espacio desempeño vs. episodios.  
Versiones: sin shaping, con shaping leve, con shaping fuerte.

In [ ]:
configs = [
    {'label': 'Kronos v1\n(sin shaping)', 'episodes': 3000, 'shaping_weight': 0.0,  'marker': 'o', 'color': '#3498db'},
    {'label': 'Kronos v2\n(shaping leve)', 'episodes': 3000, 'shaping_weight': 0.05, 'marker': 's', 'color': '#2ecc71'},
    {'label': 'Kronos v3\n(shaping fuerte)', 'episodes': 3000, 'shaping_weight': 0.15, 'marker': '^', 'color': '#e67e22'},
    {'label': 'Kronos v4\n(más episodios)', 'episodes': 6000, 'shaping_weight': 0.05, 'marker': 'D', 'color': '#9b59b6'},
]

fig, ax = plt.subplots(figsize=(9, 6))

for cfg in configs:
    print(f"Evaluando {cfg['label'].replace(chr(10),' ')}...")
    q = train_kronos(episodes=cfg['episodes'], shaping_weight=cfg['shaping_weight'],
                     save_path=None, verbose=False)
    agent = Kronos(); agent.q_table = q

    # Win rate promedio en ambos colores
    wr = []
    for kc in [-1, 1]:
        w, d, l = evaluate(agent, random_policy, n_games=N_EVAL, kronos_color=kc)
        wr.append(w / N_EVAL)
    avg_wr = np.mean(wr)

    ax.scatter(cfg['episodes'], avg_wr, marker=cfg['marker'],
               color=cfg['color'], s=180, zorder=5, label=cfg['label'])
    ax.annotate(f"{avg_wr:.2f}", (cfg['episodes'], avg_wr),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Umbral mínimo')
ax.set_title('Desempeño vs. Recursos — Variantes de Kronos\nvs. Jugador Aleatorio (promedio ambos colores)', fontsize=12)
ax.set_xlabel('Episodios de entrenamiento (recursos offline)')
ax.set_ylabel('Tasa de victorias promedio')
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('fig4_perf_vs_resources.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 4 guardada.')

---
## 5. Verificación del prerequisito mínimo

Kronos **nunca debe perder** contra el jugador aleatorio y debe ganar en al menos 50% de los casos,  
para ambos colores (rojo y amarillo).

In [ ]:
print('=== Verificación del prerrequisito mínimo ===')
final_kronos = Kronos(episodes=6000)
final_kronos.mount()  # Entrena o carga

N_CHECK = 500
for color_name, kc in [('Rojo (primer turno)', -1), ('Amarillo (segundo turno)', 1)]:
    w, d, l = evaluate(final_kronos, random_policy, n_games=N_CHECK, kronos_color=kc)
    wr = w / N_CHECK
    lr = l / N_CHECK
    ok_win  = '✅' if wr >= 0.5 else '❌'
    ok_loss = '✅' if l == 0 else f'❌ ({l} derrotas)'
    print(f"  [{color_name}]")
    print(f"    Victorias: {w}/{N_CHECK} ({wr:.1%}) {ok_win}")
    print(f"    Empates:   {d}/{N_CHECK} ({d/N_CHECK:.1%})")
    print(f"    Derrotas:  {l}/{N_CHECK} ({lr:.1%}) {ok_loss}")
    print()

---
## 6. Convergencia del aprendizaje

Evolución de la tasa de victorias durante el entrenamiento (cada 500 episodios).

In [ ]:
checkpoints = list(range(500, 6001, 500))
convergence_red, convergence_yellow = [], []
q_running = SharedQTable()

for target_ep in checkpoints:
    # Entrenar 500 episodios incrementales
    for _ in range(500):
        updates = run_self_play_episode(q_running)
        for key, col, G in updates:
            q_running.update(key, col, G)

    agent = Kronos(); agent.q_table = q_running
    w_r, _, _ = evaluate(agent, random_policy, n_games=100, kronos_color=-1)
    w_y, _, _ = evaluate(agent, random_policy, n_games=100, kronos_color=1)
    convergence_red.append(w_r / 100)
    convergence_yellow.append(w_y / 100)
    print(f'  {target_ep} ep: Rojo={w_r}% | Amarillo={w_y}%')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(checkpoints, convergence_red,    'o-', color='#e74c3c', label='Rojo (1ro)',    linewidth=2)
ax.plot(checkpoints, convergence_yellow, 's--', color='#f39c12', label='Amarillo (2do)', linewidth=2)
ax.axhline(0.5, color='gray', linestyle=':', label='50% umbral')
ax.fill_between(checkpoints, convergence_red, convergence_yellow, alpha=0.1, color='purple')
ax.set_title('Curva de convergencia de Kronos\nvs. Jugador Aleatorio', fontsize=12)
ax.set_xlabel('Episodios de entrenamiento')
ax.set_ylabel('Tasa de victorias (100 partidas)')
ax.legend()
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('fig5_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 5 guardada.')